In [0]:
# add to the fact_ball build, after the other withColumn measures:

# --- over_phase: analytical innings-phase (death → powerplay → middle → N/A) ---
# full-length from FORMAT (authoritative), scheduled_overs only to detect shortening
# (guards the 15 mislabelled T20s that have scheduled_overs=50)

from pyspark.sql import functions as F
CATALOG = "cricket"

d  = spark.table(f"{CATALOG}.silver.deliveries")
bp = spark.table(f"{CATALOG}.silver.ball_phase")
dm = spark.table(f"{CATALOG}.silver.dim_match").select(
        "match_id","start_date","event_name","season","venue",
        "match_format","scheduled_overs")          # <-- added the two phase inputs

mk = (dm
    .withColumn("date_key",   F.date_format("start_date","yyyyMMdd").cast("int"))
    .withColumn("series_key", F.xxhash64(F.concat_ws("|",
                    F.coalesce("event_name", F.lit("(no event)")), "season")))
    .withColumn("venue_key",  F.xxhash64(F.lower(F.trim(F.col("venue")))))
    # full length from FORMAT (T20->20, ODI->50); ignore for other formats
    .withColumn("full_len",
        F.when(F.col("match_format")=="T20", F.lit(20))
         .when(F.col("match_format")=="ODI", F.lit(50)))
    # sched = scheduled_overs if sane (>0 and <= full_len), else full_len
    .withColumn("sched",
        F.when((F.col("scheduled_overs") > 0) & (F.col("scheduled_overs") <= F.col("full_len")),
               F.col("scheduled_overs"))
         .otherwise(F.col("full_len")))
    # death fraction: T20 = 0.25 (last 5 of 20), ODI = 0.20 (last 10 of 50)
    .withColumn("death_frac",
        F.when(F.col("match_format")=="T20", F.lit(0.25))
         .when(F.col("match_format")=="ODI", F.lit(0.20)))
    # death starts at over index (0-indexed): sched - ceil(sched*frac)
    .withColumn("death_start_idx",
        (F.col("sched") - F.ceil(F.col("sched")*F.col("death_frac"))).cast("int"))
    .withColumn("match_format", F.col("match_format"))
    .select("match_id","date_key","series_key","venue_key",
            "match_format","death_start_idx"))

fact = (d
    .join(bp, ["match_id","innings_number","over_number","ball_seq"], "left")
    .join(mk, "match_id", "left")
    .withColumn("batting_team_key", F.xxhash64(F.lower(F.trim(F.col("batting_team")))))
    .withColumn("bowling_team_key", F.xxhash64(F.lower(F.trim(F.col("bowling_team")))))
    .withColumn("is_legal_ball", (F.col("extra_wides")==0) & (F.col("extra_noballs")==0))
    .withColumn("is_ball_faced", (F.col("extra_wides")==0))
    .withColumn("is_bowler_wicket",
        F.col("is_wicket") &
        F.col("dismissal_kind").isin("bowled","caught","caught and bowled",
                                     "lbw","stumped","hit wicket"))
    # --- over_phase: death → powerplay → middle → N/A ---
    .withColumn("over_phase",
        F.when(~F.col("match_format").isin("T20","ODI"), F.lit("N/A"))
         # death FIRST (wins the tail, incl. ODI late powerplay overlap)
         .when(F.col("over_number") >= F.col("death_start_idx"), F.lit("death"))
         # then powerplay from SOURCE (phase_key starts with 'PP')
         .when(F.col("phase_key").startswith("PP"), F.lit("powerplay"))
         # else middle
         .otherwise(F.lit("middle")))
)

In [0]:
# in the gold fact_ball build — anti-join the exclusions once, at build time
excluded = spark.table(f"{CATALOG}.silver.excluded_match").select("match_id")

# select the final star-shaped column set
fact_ball = fact.select(
    # degenerate dims (on the fact)
    "match_id", "innings_number", "over_number", "ball_seq",
    "is_super_over",
    # foreign keys
    "date_key", "series_key", "venue_key",
    "batting_team_key", "bowling_team_key",
    F.col("batter_id"), F.col("bowler_id"),
    F.col("non_striker_id"), F.col("player_out_id"),
    "phase_key",
    # measures
    "runs_batter", "runs_extras", "runs_total",
    "extra_wides", "extra_noballs", "extra_byes", "extra_legbyes", "extra_penalty",
    "is_legal_ball", "is_ball_faced", "is_wicket", "is_bowler_wicket", "wicket_count",
    "non_boundary","over_phase",
)

fact_ball = fact_ball.join(excluded, "match_id", "left_anti")   # drops voided matches
# ...then write as before

fact_ball.write.format("delta").mode("overwrite").option("overwriteSchema","true") \
    .clusterBy("match_id").saveAsTable(f"{CATALOG}.gold.fact_ball")

In [0]:
# ─────────────────────────────────────────────────────────────
# Verify — grain, FK integrity, and a real stat computed off the fact
# ─────────────────────────────────────────────────────────────
fb = spark.table(f"{CATALOG}.gold.fact_ball")

# 1) grain preserved: fact_ball == deliveries
print("fact_ball:", fb.count(), "| deliveries:", d.count())

# 2) FK integrity: no orphan keys against the dims
print("orphan date_key:",
      fb.join(spark.table(f"{CATALOG}.silver.dim_calendar"), "date_key", "left")
        .where("date is null").count())
print("null batter_id on legal balls:",
      fb.where("is_legal_ball and batter_id is null").count())

# 3) a real stat: top run-scorers (batting), joined to dim_player
spark.sql(f"""
  SELECT p.canonical_name,
         sum(f.runs_batter)                              AS runs,
         sum(CASE WHEN f.is_legal_ball THEN 1 ELSE 0 END) AS balls_faced,
         round(100.0*sum(f.runs_batter)/
               nullif(sum(CASE WHEN f.is_legal_ball THEN 1 ELSE 0 END),0),1) AS strike_rate
  FROM {CATALOG}.gold.fact_ball f
  JOIN {CATALOG}.silver.dim_player p ON f.batter_id = p.person_id
  WHERE NOT f.is_super_over
  GROUP BY p.canonical_name
  ORDER BY runs DESC LIMIT 10
""").show(truncate=False)

In [0]:
from datetime import date
CATALOG = "cricket"
row = spark.sql(f"""
  SELECT max(start_date) AS latest_match FROM {CATALOG}.gold.dim_match
""").collect()[0]
days_behind = (date.today() - row["latest_match"]).days
print(f"latest match in gold: {row['latest_match']} | {days_behind}d behind")
assert days_behind <= 5, f"DATA STALE: {days_behind} days behind — check feed/job"